# agent

## market environment

In [1]:
import numpy as np

In [2]:

class broker:
    '''
    broker agent including the holding information
    '''
    def __init__(self, market_database, transaction_cost_percent=0.5):
        '''
        database: 2D numpy array with:
            dim1 = market values of each stock
            dim2 = each time interval (day / week / month / anything else)
        
        cash: available cash
        '''
        self.db = market_database
        self.tcp = transaction_cost_percent / 100
        self.reset()
        
    def step(self, action):
        '''
        action: 
            1D array specifying the buy and sell of each stock
            make sure self.cash - sum(cost of action) >=0
            and not selling more than owned
        '''

        if action is None:
            action = np.zeros_like(self.holdings)


        if self.t >= self.db.shape[1] -1:
            # no more data available, evaluate the final holding positions
            reward = np.sum(self.holdings) + self.cash
            self.reset()
           
        else:

            # print(f"time step {self.t}")
            # print(f"legal action = {legal_actions}")
            # print(f"cash = {self.cash}")
            # print(f"holdings = {self.holdings}")

            reward = - np.sum(action)
            assert np.min(self.holdings + action) >= 0, "illegal action taken! selling more than owned"
            assert self.cash + reward >= 0, "illegal action taken! Not enough cash"

            self.holdings += action
            self.cash += reward
        
            self.t += 1
            market_change = np.divide(self.db[:, self.t], self.db[:, self.t -1])
            self.holdings = np.multiply(self.holdings, market_change)

        return reward, self.db[:, self.t], self.cash, self.holdings


    def reset(self):
        self.t = 0
        self.cash = 100
        self.holdings = np.zeros(self.db.shape[0])

## Experiments helper functions

In [3]:
# Helper functions

from functools import partial
import collections
import matplotlib.pyplot as plt


def smooth(array, smoothing_horizon=2., initial_value=0.):
  """Smoothing function for plotting."""
  smoothed_array = []
  value = initial_value
  b = 1./smoothing_horizon
  m = 1.
  for x in array:
    m *= 1. - b
    lr = b/(1 - m)
    value += lr*(x - value)
    smoothed_array.append(value)
  return np.array(smoothed_array)

def plot(algs, plot_data, repetitions=30):
  """Plot results of a bandit experiment."""
  algs_per_row = 2
  n_algs = len(algs)
  n_rows = (n_algs - 2)//algs_per_row + 1
  # algs_per_row = 2
  # n_rows = 1
  fig = plt.figure(figsize=(10, 4*n_rows))
  fig.subplots_adjust(wspace=0.3, hspace=0.35)
  clrs = ['#000000', '#00bb88', '#0033ff', '#aa3399', '#ff6600']
  lss = ['--', '-', '-', '-', '-']
  # print(plot_data)
  for i, p in enumerate(plot_data):
    for c in range(n_rows):
      ax = fig.add_subplot(n_rows, len(plot_data), i + 1 + c*len(plot_data))
      ax.grid(0)

      current_algs = [algs[0]] + algs[c*algs_per_row + 1:(c + 1)*algs_per_row + 1]
      for alg, clr, ls in zip(current_algs, clrs, lss):
        data = p.data[alg.name]
        m = smooth(np.mean(data, axis=0))
        s = np.std(smooth(data.T).T, axis=0)/np.sqrt(repetitions)
        if p.log_plot:
          line = plt.semilogy(m, alpha=0.7, label=alg.name,
                              color=clr, ls=ls, lw=3)[0]
        else:
          line = plt.plot(m, alpha=0.7, label=alg.name,
                          color=clr, ls=ls, lw=3)[0]
          plt.fill_between(range(len(m)), m + s, m - s,
                           color=line.get_color(), alpha=0.2)
      # if p.opt_values is not None:
      #   plt.plot(p.opt_values[current_algs[0].name][0], ':', alpha=0.5,
      #            label='optimal')

      ax.set_facecolor('white')
      ax.tick_params(axis="both", which="both", bottom="off", top="off",
                     labelbottom="on", left="off", right="off", labelleft="on")
      ax.spines["top"].set_visible(False)
      ax.spines["bottom"].set(visible=True, color='black', lw=1)
      ax.spines["right"].set_visible(False)
      ax.spines["left"].set(visible=True, color='black', lw=1)
      ax.get_xaxis().tick_bottom()
      ax.get_yaxis().tick_left()

      data = np.array([smooth(np.mean(d, axis=0)) for d in p.data.values()])

      if p.log_plot:
        start, end = calculate_lims(data, p.log_plot)
        start = np.floor(np.log10(start))
        end = np.ceil(np.log10(end))
        ticks = [_*10**__
                 for _ in [1., 2., 3., 5.]
                 for __ in [-2., -1., 0.]]
        labels = [r'${:1.2f}$'.format(_*10** __)
                  for _ in [1, 2, 3, 5]
                  for __ in [-2, -1, 0]]
        plt.yticks(ticks, labels)
      plt.ylim(calculate_lims(data, p.log_plot))
      plt.locator_params(axis='x', nbins=4)

      plt.title(p.title)
      if i == len(plot_data) - 1:
        plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.)

def run_experiment(bandit_constructor, algs, repetitions, number_of_steps):
  """Run multiple repetitions of a bandit experiment."""
  reward_dict = {}
  cash_dict = {}
  # optimal_value_dict = {}

  for alg in algs:
    reward_dict[alg.name] = np.zeros((repetitions, number_of_steps))
    cash_dict[alg.name] = np.zeros((repetitions, number_of_steps))
    # optimal_value_dict[alg.name] = np.zeros((repetitions, number_of_steps))

    for _rep in range(repetitions):
      bandit = bandit_constructor()
      alg.reset()

      action = None
      reward = None
      for _step in range(number_of_steps):
        reward, state, cash, holdings = bandit.step(action)
        action = alg.step(action, reward, state, cash, holdings)
        
        # regret = bandit.regret(action)
        # optimal_value = bandit.optimal_value()

        reward_dict[alg.name][_rep, _step] = reward
        cash_dict[alg.name][_rep, _step] = cash
        # optimal_value_dict[alg.name][_rep, _step] = optimal_value

  return reward_dict, cash_dict#, optimal_value_dict


def train_agents(agents, 
                #  number_of_arms,
                 market_database, 
                 number_of_steps, 
                 repetitions=100,
                #  success_reward=1., fail_reward=0.,
                 bandit_class=broker):

  # success_probabilities = np.arange(0.3, 0.7 + 1e-6, 0.4/(number_of_arms - 1))

  bandit_constructor = partial(bandit_class,
                               market_database=market_database)
  # rewards, regrets, opt_values = run_experiment(
  #     bandit_constructor, agents, repetitions, number_of_steps)
  rewards, cash = run_experiment(
      bandit_constructor, agents, repetitions, number_of_steps)
  smoothed_rewards = {}
  for agent, rs in rewards.items():
    smoothed_rewards[agent] = np.array(rs)

  PlotData = collections.namedtuple('PlotData',
                                    ['title', 'data', 'log_plot'])
  # cash = dict([(k, np.cumsum(v, axis=1)) for k, v in cash.items()])

  plot_data = [
      PlotData(title='Smoothed rewards', data=smoothed_rewards,
               log_plot=False),
      PlotData(title='Remaining cash', data=cash,
               log_plot=False),
      # PlotData(title='Total Regret', data=total_regrets, opt_values=None,
      #          log_plot=False),
  ]

  plot(agents, plot_data, repetitions)

def calculate_lims(data, log_plot=False):
  y_min = np.min(data)
  y_max = np.max(data)
  diff = y_max - y_min
  if log_plot:
    y_min = 0.9*y_min
    y_max = 1.1*y_max
  else:
    y_min = y_min - 0.05*diff
    y_max = y_max + 0.05*diff
  return y_min, y_max

def argmax(array):
  """Returns the maximal element, breaking ties randomly."""
  return np.random.choice(np.flatnonzero(array == array.max()))

In [61]:
a = np.random.random(3)
print(a)

[0.27188304 0.00572423 0.84363217]


## agents

In [18]:
class DoNothing:
    def __init__(self, name):
        self.name = name
        self.reset()
    
    def step(self, previous_action, reward, state, cash, holdings):
        return np.zeros_like(state)
    
    def reset(self):
        pass

class Random:
    '''
    Randomly allocate assets including holdings
    '''
    def __init__(self, name):
        self.name = name
        self.reset()
    
    def step(self, previous_action, reward, state, cash, holdings):

        available_assets = cash + np.sum(holdings)

        if available_assets < 0.1:
            return np.zeros_like(state)
        else:
            logits = np.random.random(state.shape)
            objective_assets = np.exp(logits) / np.sum(np.exp(logits)) * available_assets * 0.994
            return objective_assets - holdings
    
    def reset(self):
        pass

class BuyAndHoldForever:
    '''
    Randomly buy one stock and hold forever
    '''
    def __init__(self, name):
        self.name = name
        self.reset()
    
    def step(self, previous_action, reward, state, cash, holdings):
        action = np.zeros_like(state)
        if np.sum(holdings) < 0.1:
            action[np.random.randint(len(action))] = cash * 0.994
        return action
           
    def reset(self):
        pass

class LinearValueFunction:
    '''
    Use linear function to approximate value function at each time point
    use epsilon greedy for action selection
    '''
    def __init__(self, name, num_of_stocks, epsilon, lr):
        self.name = name
        self.num_stocks = num_of_stocks
        self.eps = epsilon
        self.lr = lr
        self.reset()
    
    def step(self, previous_action, reward, state, cash, holdings):
        # action = np.zeros_like(state)
        # if np.sum(holdings) < 0.1:
        #     action[np.random.randint(len(action))] = cash * 0.99
        # TODO!
        if previous_action is None:
            previous_action = np.zeros_like(state)

        TD_error = reward + 1 * np.append(state, holdings).dot(self.weights) - np.append(state, (holdings-previous_action)).dot(self.weights)
        self.weights += self.lr * np.outer(np.append(state, holdings), TD_error)
        self.weights /= np.linalg.norm((self.weights + 1e-6), ord=2)

        new_state_value = np.append(state, holdings).dot(self.weights)

        new_asset_allocation = np.zeros_like(state)
        total_asset = cash + np.sum(holdings)
        total_asset *= 0.99

        # print(f"TD_error: {TD_error}")
        # print(f"weights: {self.weights}")
        # print(f"total asset = {total_asset}")
        if total_asset < 0.1:
            return np.zeros_like(state)
        if np.random.random() <= self.eps:
            # explore
            new_asset_allocation += 1
            new_asset_allocation *= total_asset / float(len(state))
            
        else:
            new_asset_allocation[argmax(new_state_value)] = total_asset

        action = new_asset_allocation - holdings
        # print(f"action = {action}")
        return action
           
    def reset(self):
        self.weights = np.zeros((self.num_stocks*2, self.num_stocks))


class BuyLowSellHigh:
    def __init__(self, name, num_of_stocks):
        self.name = name
        self.num_of_stocks = num_of_stocks
        
        self.reset()
    
    def step(self, previous_action, reward, state, cash, holdings):
        self.previous_high = np.maximum(self.previous_high, state)
        total_asset = cash + np.sum(holdings)
        action = np.zeros_like(state)
        if total_asset < 0.1:
            # no asset available, do nothing
            pass
            
        elif np.sum(holdings) < 0.1:
            # no asset purchased
            quotient = np.divide(state, self.previous_high)
            if len(quotient[quotient < 0.5]) > 0:
                self.bought_stock = argmax(-quotient)
                action[self.bought_stock] = total_asset * 0.99
                
                self.bought_price = state[self.bought_stock]
                
        else:
            # already in a position
            quotient = state[self.bought_stock] / self.bought_price
            if quotient >=2.5:
                action[self.bought_stock] = - holdings[self.bought_stock]
                self.bought_price = 0
                self.bought_stock = None
        
        return action
    
    def reset(self):
        self.previous_high = np.zeros_like(self.num_of_stocks)
        self.bought_price = 0
        self.bought_stock = None
        
class BuyLowNoSell:
    def __init__(self, name, num_of_stocks):
        self.name = name
        self.num_of_stocks = num_of_stocks
        
        self.reset()
    
    def step(self, previous_action, reward, state, cash, holdings):
        self.previous_high = np.maximum(self.previous_high, state)
        total_asset = cash + np.sum(holdings)
        action = np.zeros_like(state)
        if cash > 90:
            # no asset purchased
            quotient = np.divide(state, self.previous_high)
            if len(quotient[quotient < 0.7]) > 0:
                self.bought_stock = argmax(-quotient)
                action[self.bought_stock] = total_asset * 0.99
        
        return action
    
    def reset(self):
        self.previous_high = np.zeros_like(self.num_of_stocks)

class BuyAllAndHoldForever:
    '''
    Buy all stocks and hold forever
    '''
    def __init__(self, name):
        self.name = name
        self.reset()
    
    def step(self, previous_action, reward, state, cash, holdings):
        total_asset = cash + np.sum(holdings)
        action = np.zeros_like(state)
        if cash > 0.2:
            action += 1 
            action *= cash / len(state) * 0.99
        if total_asset < 0.1:
            pass
        return action
           
    def reset(self):
        pass


from models import MLP_predictor
import torch

class MLGreedy:
    def __init__(self, name):
        
        self.name = name
        self.device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

        self.predictor = MLP_predictor(7).to(self.device)
        self.predictor.load_state_dict(torch.load("predict_real_market.pt"))
        self.reset()
    
    def step(self, previous_action, reward, state, cash, holdings):
        target_asset = np.zeros_like(state)
        total_asset = cash + np.sum(holdings) * 0.99
        prediction = self.predictor(torch.tensor(state.astype(np.float32)).to(self.device)).detach().cpu().numpy()
        percentage_change = prediction / state
        # if np.max(percentage_change) > 1.2:
        #     target_asset[argmax(percentage_change)] = total_asset
        # elif np.max(percentage_change) > 1:
        #     target_asset = holdings
        
        target_asset[argmax(prediction)] = total_asset
        action = target_asset - holdings
        return action
    
    def reset(self):
        pass
        

In [62]:
print(len(a[a<0.2]))

1


In [19]:
number_of_steps = 1e3


time_points = 200
number_stocks = 50
fluctuation_range = 0.2 * np.random.random(number_stocks)
# print(fluctuation_range)
# exponent_end = 2.
# interval = exponent_end / time_points

market_database = np.ones((number_stocks, time_points))
time_point_fluctuation = np.random.random((number_stocks, time_points)) 
for i in range(market_database.shape[1] -1):
    market_database[:, i+1] *= (1.0000 - fluctuation_range/2 + time_point_fluctuation[:, i] * fluctuation_range) * market_database[:, i]

print(f"expected average return is {np.mean(market_database[:, -1])}")


agents = [
    DoNothing(
        "DoNothing"
    ),
    BuyAndHoldForever(
        "BuyAndHoldForever"
    ),
    # Random(
    #     "Random"
    # ),
    # LinearValueFunction(
    #     name="LinearValueFunction",
    #     num_of_stocks=number_stocks,
    #     epsilon=0.1,
    #     lr=0.1
    # ),

    BuyLowSellHigh(
        name="BuyLowSellHigh",
        num_of_stocks=number_stocks
    ),

    BuyLowNoSell(
        name="BuyLowNoSell",
        num_of_stocks=number_stocks
    ),
    MLGreedy(
        name="ML Greedy",
    ),

    BuyAllAndHoldForever(
        name="BuyAllAndHoldForever"
    ),

]

train_agents(agents, market_database, int(number_of_steps))
plt.show()

for i in range(market_database.shape[0]):
    plt.plot(market_database[i,:], label=f"stock {i}")
plt.title("stock_movement")
plt.legend()
plt.show()

expected average return is 0.8930220235580105


IndexError: Dimension out of range (expected to be in range of [-1, 0], but got -2)

## use real market data

In [13]:
from data_utils import get_csv_list, batch_load
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


market_df = batch_load(get_csv_list("./data"), pd_df=True, normalise=True)
stock_names = market_df.columns
sp500_idx = stock_names.get_loc("sp500")

optimal_chance = 0.3
transaction_cost = 0.5
total_time_points = 500

market_database = market_df.to_numpy().transpose()

if total_time_points >= 1:
    market_database = market_database[:, :total_time_points]
print(market_database.shape)
norm_market_database = np.zeros_like(market_database)
for i in range(market_database.shape[0]):
    norm_market_database[i, :] = market_database[i, :] / market_database[i, -1]




(7, 500)


In [15]:
number_stocks = 7

agents = [
    DoNothing(
        "DoNothing"
    ),
    BuyAndHoldForever(
        "BuyAndHoldForever"
    ),
    Random(
        "Random"
    ),
    LinearValueFunction(
        name="LinearValueFunction",
        num_of_stocks=number_stocks,
        epsilon=0.1,
        lr=0.1
    ),

    BuyLowSellHigh(
        name="BuyLowSellHigh",
        num_of_stocks=number_stocks
    ),

    BuyLowNoSell(
        name="BuyLowNoSell",
        num_of_stocks=number_stocks
    ),
    MLGreedy(
        name="ML Greedy",
    ),

    BuyAllAndHoldForever(
        name="BuyAllAndHoldForever"
    ),

]

train_agents(agents, norm_market_database, norm_market_database.shape[-1])
plt.show()

RuntimeError: Error(s) in loading state_dict for MLP_predictor:
	Unexpected key(s) in state_dict: "layers.4.bias", "layers.4.parametrizations.weight.original0", "layers.4.parametrizations.weight.original1", "layers.6.bias", "layers.6.parametrizations.weight.original0", "layers.6.parametrizations.weight.original1". 
	size mismatch for layers.2.bias: copying a param with shape torch.Size([256]) from checkpoint, the shape in current model is torch.Size([7]).
	size mismatch for layers.2.parametrizations.weight.original0: copying a param with shape torch.Size([256, 1]) from checkpoint, the shape in current model is torch.Size([7, 1]).
	size mismatch for layers.2.parametrizations.weight.original1: copying a param with shape torch.Size([256, 256]) from checkpoint, the shape in current model is torch.Size([7, 256]).